# YOLOv8 — avaliação de todos os modelos e ranking final

Avalia os experimentos originais e os modelos com preprocessamento disponíveis.
Modelos sem `result.json` ou `best.pt` são pulados. Não treina redes.
Cada variante recebe exatamente o filtro e os parâmetros usados no treinamento,
inclusive a referência original da grade de preprocessamento, exportada como PNG.

**Padrão: execução local**, onde estão os 11 modelos treinados e o exp_05 importado.
Para incluir os outros modelos originais, disponibilize suas pastas nos caminhos
configurados abaixo. O notebook não acessa remotamente arquivos ausentes.
Também funciona no Colab; nesse caso, os modelos locais precisam estar no Drive.

O ranking final usa **mAP50–95 no teste**, em ordem decrescente. A escolha pela
validação é registrada antes da avaliação e permanece indicada separadamente.
O ranking de teste é descritivo: não reutilize o teste para ajustar filtros ou
hiperparâmetros. As duas grades têm batches e aumentos diferentes; diferenças
entre elas não podem ser atribuídas exclusivamente ao preprocessamento.


## 1. Ambiente e caminhos
Selecione `local` ou `colab` antes de executar.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

MODE = "local"  # 'local' ou 'colab'
DEVICE = "0"  # 'cpu' se não houver GPU disponível
BATCH = 1

if MODE == "local":
    project = next(
        (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file()),
        None,
    )
    if project is None:
        raise RuntimeError("Abra o notebook a partir da pasta do repositório.")
    sys.path.insert(0, str(project))
    from src.paths import storage_root

    storage = storage_root()
    executable = "Scripts/python.exe" if os.name == "nt" else "bin/python"
    PYTHON = [str(storage / ".venv" / executable)]
    GRIDS = [storage / "runs/grid", storage / "models"]
    PREPROCESSING_GRID = storage / "runs/preprocessing_local/grid"
    OUTPUT = storage / "runs/final_test_all"
    CACHE = storage / "evaluation_test_cache"
elif MODE == "colab":
    project = Path("/content/what-is-the-book-2")
    storage = Path("/content/drive/MyDrive/what-is-the-book-2")
    PYTHON = ["uv", "run", "--no-sync", "python"]
    GRIDS = [storage / "runs/grid"]
    PREPROCESSING_GRID = storage / "runs/preprocessing_local/grid"
    OUTPUT = storage / "runs/final_test_all"
    CACHE = Path("/content/evaluation_test_cache")
else:
    raise ValueError("MODE deve ser local ou colab")

# Ajuste PREPROCESSING_GRID caso a pasta tenha outro nome no Drive.
# Use None apenas para avaliar exclusivamente a grade original.
DATA = project / "data/unified"
os.environ["MPLBACKEND"] = "Agg"

## 2. Preparação
Local: usa `.venv` no armazenamento configurado, preservando CUDA.
Colab: monta o Drive, solicita o ZIP se o dataset não estiver presente e instala
o ambiente isolado. Use um ZIP atualizado que inclua `src/preprocess/image_variants.py`.
Não execute outra avaliação na mesma pasta nem concorra com um treinamento na GPU.


In [ ]:
if MODE == "colab":
    import zipfile

    from google.colab import drive, files

    drive.mount("/content/drive")
    if not (DATA / "manifest.json").is_file():
        uploaded = files.upload()
        assert "colab_bundle.zip" in uploaded, "Selecione colab_bundle.zip"
        project.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile("colab_bundle.zip") as archive:
            assert all(
                (project / name).resolve().is_relative_to(project.resolve())
                for name in archive.namelist()
            )
            archive.extractall(project)
        del uploaded
    os.chdir(project)
    subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
    subprocess.run(
        ["uv", "sync", "--locked", "--python", "3.13", "--extra", "training", "--no-dev"],
        check=True,
    )
else:
    assert Path(PYTHON[0]).is_file(), "Ambiente local não encontrado"
    os.chdir(project)
assert (DATA / "manifest.json").is_file(), "Dataset não encontrado"
subprocess.run(
    PYTHON
    + [
        "-c",
        "import torch; print(torch.__version__); "
        'print("CUDA disponível:", torch.cuda.is_available())',
    ],
    check=True,
)

## 3. Conferir cobertura
Somente checkpoints acessíveis nesta máquina serão avaliados. Na configuração
local atual, espere 11 variantes mais o exp_05. Para comparar todos os originais,
inclua também seus `best.pt` e `result.json`, preservando a estrutura `exp_XX/`.
São aceitos `exp_XX/weights/best.pt` ou `exp_XX/best.pt`.


In [ ]:
ready = []
for i in range(1, 9):
    name = f"exp_{i:02d}"
    found = any(
        (root / name / "result.json").is_file()
        and any((root / name / p).is_file() for p in ["weights/best.pt", "best.pt"])
        for root in GRIDS
    )
    print(name, "pronto" if found else "será pulado: incompleto ou indisponível")
    if found:
        ready.append(name)
if PREPROCESSING_GRID is not None:
    assert PREPROCESSING_GRID.is_dir(), f"Pasta não encontrada: {PREPROCESSING_GRID}"
    for folder in sorted(PREPROCESSING_GRID.iterdir()):
        if folder.is_dir():
            complete = (folder / "result.json").is_file() and (folder / "weights/best.pt").is_file()
            print("preproc__" + folder.name, "pronto" if complete else "será pulado")
            if complete:
                ready.append("preproc__" + folder.name)
print("Total disponível:", len(ready))
print("Relatórios:", OUTPUT)

## 4. Avaliador
Valida a identidade dos dados, versões e checkpoints. Para variantes, valida
também o código de preprocessamento e gera somente as imagens de teste em PNG,
com anotações idênticas às originais. Remove cada cache gerado após salvar o resultado.
Não altera `data/unified`. A resolução é a usada no treinamento de cada modelo.


In [ ]:
EVALUATOR = r'''
"""Evaluate original and preprocessing YOLOv8 experiments on the held-out test set."""

import argparse
import csv
import hashlib
import importlib
import json
import os
from pathlib import Path

import cv2
import numpy as np

from src.preprocess import image_variants


def digest(path: Path) -> str:
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def save_json(path: Path, value: object) -> None:
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")
    temporary.replace(path)


def inspect_data(data: Path) -> tuple[str, str, int]:
    manifest_path = data / "manifest.json"
    entries = json.loads(manifest_path.read_text(encoding="utf-8"))
    groups: dict[str, str] = {}
    test = []
    for entry in entries:
        group = str(entry["group"])
        split = entry["split"]
        if group in groups and groups[group] != split:
            raise ValueError(f"Group {group} appears in multiple splits")
        groups[group] = split
        if split == "test":
            test.append(entry)
    if not test:
        raise ValueError("Test split is empty")
    signature = hashlib.sha256()
    expected = set()
    for entry in sorted(test, key=lambda item: item["image"]):
        image = (data / entry["image"]).resolve()
        if not image.is_relative_to((data / "test/images").resolve()):
            raise ValueError("Invalid test image path")
        label = data / "test/labels" / (image.stem + ".txt")
        expected.add(image)
        signature.update(entry["image"].encode())
        signature.update(bytes.fromhex(digest(image)))
        signature.update(bytes.fromhex(digest(label)))
    actual = {p.resolve() for p in (data / "test/images").iterdir() if p.is_file()}
    if actual != expected:
        raise ValueError("Test image directory differs from the manifest")
    return digest(manifest_path), signature.hexdigest(), len(test)


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data", type=Path, required=True)
    parser.add_argument("--grid", type=Path, action="append", default=[])
    parser.add_argument("--preprocessing-grid", type=Path)
    parser.add_argument("--cache", type=Path)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--device", default="cpu")
    parser.add_argument("--batch", type=int, default=1)
    args = parser.parse_args()
    if args.batch < 1:
        parser.error("Batch must be positive")
    data, output = args.data.resolve(), args.output.resolve()
    grids = [path.resolve() for path in args.grid]
    preprocessing = args.preprocessing_grid
    if not grids and preprocessing is None:
        parser.error("Provide --grid and/or --preprocessing-grid")
    if output in grids or output.is_relative_to(data):
        raise ValueError("Use a separate output directory outside the dataset")
    manifest_hash, test_hash, count = inspect_data(data)
    experiments = []
    skipped = {}
    for number in range(1, 9) if grids else []:
        name = f"exp_{number:02d}"
        candidates = []
        for grid in grids:
            folder = grid / name
            result_path = folder / "result.json"
            best = folder / "weights/best.pt"
            if not best.is_file():
                best = folder / "best.pt"  # Compact exports downloaded from Drive.
            if result_path.is_file() and best.is_file():
                candidates.append((result_path, best))
        if not candidates:
            skipped[name] = "Missing result.json or best.pt in configured original grids"
            print(f"STATUS | {name}: incomplete or unavailable; skipping", flush=True)
            continue
        result_path, best = candidates[0]
        if any(digest(other) != digest(best) for _, other in candidates[1:]):
            raise ValueError(f"{name}: conflicting checkpoints in configured grids")
        result = json.loads(result_path.read_text(encoding="utf-8"))
        if result["dataset_sha256"] != manifest_hash:
            raise ValueError(f"{name}: dataset differs from the training manifest")
        result["family"] = "original_grid"
        experiments.append((name, result, best))
    if preprocessing is not None:
        if not preprocessing.is_dir():
            raise FileNotFoundError(f"Preprocessing grid not found: {preprocessing}")
        _, train_val_hash = image_variants.dataset_identity(data)
        implementation_hash = digest(Path(image_variants.__file__))
        for folder in sorted(preprocessing.iterdir()):
            if not folder.is_dir():
                continue
            name = f"preproc__{folder.name}"
            result_path, best = folder / "result.json", folder / "weights/best.pt"
            if not result_path.is_file() or not best.is_file():
                skipped[name] = "Missing result.json or weights/best.pt"
                print(f"STATUS | {name}: incomplete; skipping", flush=True)
                continue
            saved_result = json.loads(result_path.read_text(encoding="utf-8"))
            request = saved_result["request"]
            if request["train_val_sha256"] != train_val_hash:
                raise ValueError(f"{name}: train/validation dataset changed")
            if request["preprocessing_sha256"] != implementation_hash:
                raise ValueError(f"{name}: preprocessing implementation changed")
            if request["opencv"] != cv2.__version__ or request["numpy"] != np.__version__:
                raise ValueError(f"{name}: install the OpenCV/NumPy versions used in training")
            result = {**request, "metrics": saved_result["metrics"], "family": "preprocessing"}
            experiments.append((name, result, best))
    output.mkdir(parents=True, exist_ok=True)
    save_json(output / "skipped_experiments.json", skipped)
    if not experiments:
        raise RuntimeError("No completed experiments available; nothing to evaluate")
    print(f"STATUS | Evaluating {len(experiments)} completed experiments", flush=True)
    selected = max(experiments, key=lambda e: e[1]["metrics"]["metrics/mAP50-95(B)"])[0]
    os.environ["MPLBACKEND"] = "Agg"
    ultra = importlib.import_module("ultralytics")
    torch = importlib.import_module("torch")
    for name, result, _ in experiments:
        if result["ultralytics"] != ultra.__version__:
            raise ValueError(f"{name}: install ultralytics=={result['ultralytics']}")
    output.mkdir(parents=True, exist_ok=True)
    plan = {
        "selected_by_validation": selected,
        "skipped_experiments": skipped,
        "manifest_sha256": manifest_hash,
        "test_sha256": test_hash,
        "test_images": count,
        "ultralytics": ultra.__version__,
        "torch": torch.__version__,
        "device": args.device,
        "batch": args.batch,
        "conf": 0.001,
        "iou": 0.7,
        "max_det": 1000,
        "half": False,
        "augment": False,
        "checkpoints": {name: digest(best) for name, _, best in experiments},
        "settings": {name: result["settings"] for name, result, _ in experiments},
        "variants": {name: result.get("variant") for name, result, _ in experiments},
        "validation_metrics": {name: result["metrics"] for name, result, _ in experiments},
        "opencv": cv2.__version__,
        "numpy": np.__version__,
        "preprocessing_sha256": digest(Path(image_variants.__file__)),
        "evaluator_sha256": digest(Path(__file__)),
    }
    plan_path = output / "evaluation_plan.json"
    if plan_path.exists() and json.loads(plan_path.read_text(encoding="utf-8")) != plan:
        raise ValueError("Evaluation configuration changed; choose a new output directory")
    save_json(plan_path, plan)  # Selection is frozen before any test inference.
    yaml = output / "test_dataset.yaml"
    yaml.write_text(
        f"path: {json.dumps(data.as_posix())}\ntrain: train/images\nval: val/images\n"
        "test: test/images\nnames: [book]\nnc: 1\n",
        encoding="utf-8",
    )
    rows = []
    cache = (args.cache or output / "generated_test").resolve()
    if cache == data or cache.is_relative_to(data):
        raise ValueError("Generated test cache must be outside the source dataset")
    test_entries = [
        item
        for item in json.loads((data / "manifest.json").read_text(encoding="utf-8"))
        if item["split"] == "test"
    ]
    for name, result, best in experiments:
        destination = output / name
        destination.mkdir(exist_ok=True)
        summary = destination / "test_metrics.json"
        if summary.exists():
            saved = json.loads(summary.read_text(encoding="utf-8"))
            print(f"STATUS | {name}: test already evaluated; skipping", flush=True)
        else:
            print(f"STATUS | {name}: evaluating {count} test images", flush=True)
            evaluation_yaml = yaml
            generated = None
            if "variant" in result:
                generated = cache / name
                image_variants.prepare_variant(
                    data, generated, test_entries, result["variant"], plan
                )
                evaluation_yaml = destination / "test_dataset.yaml"
                evaluation_yaml.write_text(
                    f"path: {json.dumps(generated.as_posix())}\n"
                    "train: test/images\nval: test/images\ntest: test/images\n"
                    "names: [book]\nnc: 1\n",
                    encoding="utf-8",
                )
            metrics = ultra.YOLO(str(best)).val(
                data=str(evaluation_yaml),
                split="test",
                imgsz=result["settings"]["imgsz"],
                batch=args.batch,
                device=args.device,
                workers=0,
                conf=0.001,
                iou=0.7,
                max_det=1000,
                half=False,
                augment=False,
                plots=True,
                project=str(destination),
                name="plots",
                exist_ok=True,
                verbose=False,
            )
            saved = {
                "metrics": {k: float(v) for k, v in metrics.results_dict.items()},
                "speed_ms_per_image": {k: float(v) for k, v in metrics.speed.items()},
            }
            save_json(summary, saved)
            if generated is not None:
                image_variants.release_variant(generated, cache, plan)
        rows.append(
            {
                "experiment": name,
                "family": result["family"],
                "preprocessing": result.get("variant", {}).get("name", "none"),
                "selected_by_validation": name == selected,
                "imgsz": result["settings"]["imgsz"],
                "lr0": result["settings"]["lr0"],
                "mosaic": result["settings"]["mosaic"],
                "training_batch": result["settings"].get("batch"),
                "val_mAP50_95": result["metrics"]["metrics/mAP50-95(B)"],
                "test_precision": saved["metrics"]["metrics/precision(B)"],
                "test_recall": saved["metrics"]["metrics/recall(B)"],
                "test_mAP50": saved["metrics"]["metrics/mAP50(B)"],
                "test_mAP50_95": saved["metrics"]["metrics/mAP50-95(B)"],
            }
        )
        save_json(output / "comparison.json", rows)
        with (output / "comparison.csv").open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
    ranked = sorted(rows, key=lambda row: (-row["test_mAP50_95"], row["experiment"]))
    ranking = [{"rank": i, **row} for i, row in enumerate(ranked, 1)]
    save_json(output / "ranking.json", ranking)
    with (output / "ranking.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(ranking[0]))
        writer.writeheader()
        writer.writerows(ranking)
    for row in ranking:
        print(
            f"STATUS | Rank {row['rank']}: {row['experiment']} "
            f"test mAP50-95={row['test_mAP50_95']:.4f}",
            flush=True,
        )
    print(f"STATUS | Finished. Model selected on validation: {selected}. Reports: {output}")


if __name__ == "__main__":
    main()
'''
worker = project / "evaluate_final_test.py"
worker.write_text(EVALUATOR, encoding="utf-8")

## 5. Executar avaliação
Salva progresso e logs completos. Reexecuções idênticas reutilizam métricas já
concluídas; uma avaliação interrompida é refeita apenas para o modelo pendente.
Ao incluir novos modelos ou alterar a configuração, escolha outra pasta OUTPUT.
O notebook mostra uma mensagem por modelo, sem imprimir todos os lotes.


In [ ]:
from datetime import datetime

OUTPUT.mkdir(parents=True, exist_ok=True)
log = OUTPUT / f"evaluation_{datetime.now():%Y%m%d_%H%M%S}.log"
command = PYTHON + [
    "-u",
    str(worker),
    "--data",
    str(DATA),
    "--output",
    str(OUTPUT),
    "--cache",
    str(CACHE),
    "--device",
    DEVICE,
    "--batch",
    str(BATCH),
]
for grid in GRIDS:
    command += ["--grid", str(grid)]
if PREPROCESSING_GRID is not None:
    command += ["--preprocessing-grid", str(PREPROCESSING_GRID)]
print("Log:", log, flush=True)
with log.open("w", encoding="utf-8") as handle:
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        env={**os.environ, "PYTHONIOENCODING": "utf-8"},
    )
    try:
        for line in process.stdout:
            handle.write(line)
            handle.flush()
            if "STATUS |" in line:
                print(line.strip(), flush=True)
        code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
if code:
    print("".join(log.read_text(encoding="utf-8").splitlines(keepends=True)[-35:]))
    raise RuntimeError(f"Avaliação interrompida (código {code}); consulte o log.")

## 6. Ranking final no teste
Ordenado por mAP50–95 decrescente. ★ identifica a escolha feita **pela validação**,
não pelo teste. São salvos `ranking.csv`, `ranking.json` e os arquivos de comparação.
Precisão e recall correspondem ao ponto retornado pelo validador; o limiar baixo
`conf=0.001` permite calcular as curvas AP e não é uma recomendação de uso no app.


In [ ]:
import json

from IPython.display import Markdown, display

ranking = json.loads((OUTPUT / "ranking.json").read_text(encoding="utf-8"))
plan = json.loads((OUTPUT / "evaluation_plan.json").read_text(encoding="utf-8"))
print("Imagens de teste:", plan["test_images"])
print("Escolhido pela validação:", plan["selected_by_validation"])
print("Experimentos pulados:", plan["skipped_experiments"])
table = [
    "| Rank | Modelo | Família | Resolução | Val mAP50–95 | "
    "Teste P | Teste R | Teste mAP50 | Teste mAP50–95 |",
    "|---:|---|---|---:|---:|---:|---:|---:|---:|",
]
for row in ranking:
    name = row["experiment"] + (" ★" if row["selected_by_validation"] else "")
    keys = ["val_mAP50_95", "test_precision", "test_recall", "test_mAP50", "test_mAP50_95"]
    values = " | ".join(f"{100 * row[key]:.2f}%" for key in keys)
    table.append(f"| {row['rank']} | {name} | {row['family']} | {row['imgsz']} | {values} |")
display(Markdown("\n".join(table)))
print("Ranking CSV:", OUTPUT / "ranking.csv")

## 7. Curvas e exemplos
Por padrão mostra o modelo escolhido pela validação. Altere `SHOW_MODEL` para
qualquer identificador exibido no ranking. Imagens de variantes mostram a entrada
processada usada na avaliação. Os gráficos completos ficam em `<modelo>/plots`.


In [ ]:
from IPython.display import Image

SHOW_MODEL = plan["selected_by_validation"]
plots = OUTPUT / SHOW_MODEL / "plots"
for pattern in [
    "*PR_curve.png",
    "confusion_matrix_normalized.png",
    "*batch1_labels.jpg",
    "*batch1_pred.jpg",
]:
    for image_path in sorted(plots.glob(pattern)):
        print(image_path.name)
        display(Image(filename=str(image_path), width=900))